# CS 336 playground

*  TODO look up "min p sampling"

In [ ]:
# from the 7.2 secion

vocab_size = 10000
context_length = 256
d_model = 512
d_ff = 1344
rope_theta = 10000
num_layers = 4
num_heads = 16


In [2]:
import torch

## BPE training

In [3]:
from cs336_basics.ron_train_bpe import train_bpe
"""
    vocab size of 1000 on TinyStoriesV2-GPT4-valid.txt  
    isn't horrible for CPU-based interactive notebook use.
    It takes 16 seconds to train and makes words like " Sally" and " helped" 
    into single tokens. Maybe roughly the vocab of a preschooler :)

    vocab size of 10000 on TinyStoriesV2-GPT4-valid.txt takes like 
    three minutes. and makes words like " sinking" and " ribbit".
"""
if regenerate_vocab := False:
    vocab,merges = train_bpe('../data/TinyStoriesV2-GPT4-valid.txt',10000,[])
    obj = {"vocab": vocab,"merges": merges}
    torch.save(obj, f"/tmp/bpe_{vocab_size}.saved")
else:
    obj = torch.load(f"/tmp/bpe_{vocab_size}.saved")
    vocab,merges = obj['vocab'],obj['merges']
print(merges[-5:])

[(b' s', b'ly'), (b' s', b'inking'), (b' ro', b'les'), (b' ro', b'amed'), (b' ribb', b'it')]


## BPE class

In [4]:
from cs336_basics.ron_bpe_tokenizer import RonBPETokenizer
tokenizer = RonBPETokenizer(vocab,merges)
tokens    = list(tokenizer.encode_iterable(["Hello world"," ","Good bye"]))
print(tokens)
print(tokenizer.decode(tokens))

[1207, 1597, 32, 2098, 5452]
Hello world Good bye


## Make a nice training dataset

In [5]:
import numpy as np

def tokenize_file_to_numpy(in_path, out_path, tokenizer, dtype=np.uint16):
    tokens = []
    with open(out_path, "ab") as fout:
        with open(in_path, "r", encoding="utf-8") as fin:
            for line in fin:
                toks = tokenizer.encode(line)
                toks_np = np.asarray(toks, dtype=dtype)
                fout.write(toks_np.tobytes())

# about 7 seconds with a 1000 vocab
# about 30 seconds with a 10000 vocab
if regen_numpy_ds:=False:
    tokenize_file_to_numpy('../data/TinyStoriesV2-GPT4-valid.txt',
                        "/tmp/tiny_stories_validation.uint16",
                        tokenizer)

In [6]:
token_ds = np.memmap("/tmp/tiny_stories_validation.uint16", dtype=np.uint16, mode="r")
token_ds

memmap([117, 865, 498, ..., 384, 382,  46],
       shape=(39685408,), dtype=uint16)

## TRAIN!!!

In [7]:
import cs336_basics.ron_transformer_lm as ron_transformer_lm

# d_model = 512
# num_heads = 8
# d_ff = 128
# max_seq_len = 1000
# rope_theta = 10000
# vocab_size = 1000
# context_length = 1000
# num_layers = 4

tlm = ron_transformer_lm.TransformerLM(
    d_model=d_model,
    num_heads=num_heads,
    d_ff = d_ff,
    max_seq_len=context_length,
    theta=rope_theta,
    vocab_size=vocab_size,
    context_length=context_length,
    num_layers=num_layers
    )
tlm.to('cuda')
tlm.forward([tokenizer.encode("Hello World")])

tensor([[[-0.4596, -0.2839, -0.0836,  ...,  0.2292,  0.0550,  0.4907],
         [ 0.1051,  0.2975,  0.2255,  ..., -0.3243, -0.0008,  0.7411],
         [-0.0411, -0.2556,  0.2493,  ..., -0.0870, -0.1917,  0.5119],
         [-0.0131,  0.1571,  0.3230,  ...,  0.1249,  0.0647,  0.2996]]],
       device='cuda:0', grad_fn=<ViewBackward0>)

In [8]:
import cs336_basics.ron_adamw_optimizer as ron_adamw_optimizer
my_adamw = ron_adamw_optimizer.AdamW(tlm.parameters())

In [9]:
from cs336_basics.ron_data_loader import get_batch

def get_training_batch():
    return get_batch(token_ds,10,context_length,'cpu')
get_training_batch()


(tensor([[ 435,  365, 1441,  ...,  324,  428,  488],
         [  46,  341,  701,  ...,  266,  509,  449],
         [  44,  318,   67,  ...,  495,  414,  405],
         ...,
         [ 375,  373,   10,  ..., 2710,  944,  340],
         [  44,  417,  350,  ...,  283,  834,  272],
         [  46,  317,  614,  ...,  350,  865,  498]], dtype=torch.uint16),
 tensor([[ 365, 1441, 2381,  ...,  428,  488,  283],
         [ 341,  701,  389,  ...,  509,  449,  464],
         [ 318,   67,  301,  ...,  414,  405,   10],
         ...,
         [ 373,   10, 8486,  ...,  944,  340, 1123],
         [ 417,  350,  595,  ...,  834,  272,  978],
         [ 317,  614,  258,  ...,  865,  498,  663]], dtype=torch.uint16))

In [14]:
import torch
import torch.nn.functional as F
import time
def train(model, optimizer, get_batch, device="cuda", 
          num_iters=200):
    model.to(device)
    model.train()
    t0 = time.time()
    for it in range(num_iters):
        x, y = get_batch()
        x = torch.tensor(x, dtype=torch.long, device=device)
        y = torch.tensor(y, dtype=torch.long, device=device)

        logits = model(x)            # shape: (batch, seq, vocab)
        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            y.view(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if it % 10 == 0:
            print(f"iter {it} | loss {loss.item():.4f} in {time.time() - t0} secs")

train(tlm,my_adamw,get_training_batch)


iter 0 | loss 4.8390 in 0.13590669631958008 secs


/tmp/ipykernel_2342740/1863312559.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.long, device=device)
/tmp/ipykernel_2342740/1863312559.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y = torch.tensor(y, dtype=torch.long, device=device)


iter 10 | loss 4.5950 in 1.2555410861968994 secs
iter 20 | loss 4.2459 in 2.364619255065918 secs
iter 30 | loss 4.0452 in 3.478226900100708 secs
iter 40 | loss 3.8064 in 4.595143556594849 secs
iter 50 | loss 3.6955 in 5.716746091842651 secs
iter 60 | loss 3.5169 in 6.84218692779541 secs
iter 70 | loss 3.7647 in 7.966413736343384 secs
iter 80 | loss 3.5909 in 9.0914466381073 secs
iter 90 | loss 3.6783 in 10.216469764709473 secs
iter 100 | loss 3.3042 in 11.339251041412354 secs
iter 110 | loss 3.4304 in 12.482167720794678 secs
iter 120 | loss 3.6112 in 13.598652839660645 secs
iter 130 | loss 3.5073 in 14.715491771697998 secs
iter 140 | loss 3.2186 in 15.832973718643188 secs
iter 150 | loss 3.1683 in 16.949625730514526 secs
iter 160 | loss 3.1402 in 18.066181659698486 secs
iter 170 | loss 3.2856 in 19.182992458343506 secs
iter 180 | loss 2.9667 in 20.29800271987915 secs
iter 190 | loss 3.1583 in 21.413622617721558 secs


In [15]:
import torch
predictions = tlm.forward([tokenizer.encode("Hello World")])
predictions

tensor([[[-7.2034, -6.6378, -7.1759,  ..., -7.0658, -7.1428, -5.9566],
         [-3.8771, -3.4849, -3.7509,  ..., -4.7403, -3.8843, -3.1552],
         [-5.3639, -4.7086, -4.8230,  ..., -5.4274, -5.2908, -4.3064],
         [-7.0525, -6.4369, -7.0343,  ..., -7.2249, -6.9382, -6.6548]]],
       device='cuda:0', grad_fn=<ViewBackward0>)

In [16]:
last_logits = predictions[0, -1]
next_token_id = last_logits.argmax().item()
tokenizer.decode([next_token_id])

'.'

In [19]:
num_tokens_to_generate = 50

# Convert to a list for easy appending
generated_tokens = tokenizer.encode("The boy and")

for _ in range(num_tokens_to_generate):
    seq_len = len(generated_tokens)
    tokens_tensor = torch.tensor([generated_tokens])

    with torch.no_grad():
        predictions = tlm.forward(tokens_tensor)  # (1, seq_len, vocab_size)

    last_logits = predictions[0, -1]
    next_token_id = last_logits.argmax().item()
    generated_tokens.append(next_token_id)

# Decode the whole sequence
decoded_text = tokenizer.decode(generated_tokens)
print(decoded_text)


The boy and the cat became friends.
<|endoftext|>


One day, a little girl named Lily went to the park. She saw a big tree with her friends. She wanted to play with her friends. She had a big tree. She had
